# 06 - 事件数据采集

本 Notebook 完成以下任务：
1. 股东增减持数据采集 → `stock_holder_trade`
2. 股东户数数据采集 → `stock_holder_count`
3. 数据验证

---

## 设计方案

| 数据 | Tushare 接口 | 更新频率 | 说明 |
|------|-------------|---------|------|
| 股东增减持 | stk_holdertrade | 不定期 | 按股票拉，含股东名称/类型/增减持量 |
| 股东户数 | stk_holdernumber | 季度 | 按股票拉，含股东总数和变化率 |

这两类数据没有固定的"交易日"维度，采用按股票逐只拉取的方式。

## 1. 股东增减持

In [ ]:
from invest_model.db import get_engine
from invest_model.sources.tushare_client import TushareClient
from invest_model.collectors.event_collector import EventCollector
from invest_model.repositories.stock_pool_repo import StockPoolRepository

engine = get_engine()
ts_client = TushareClient()

pool_repo = StockPoolRepository(engine)
stock_codes = pool_repo.get_pool_codes("core")
print(f"待采集: {stock_codes}")

collector = EventCollector(ts_client, engine)
n = collector.collect_holder_trade(stock_codes)
print(f"股东增减持采集完成: {n} 条")

17:24:44 | INFO    | Tushare 客户端初始化完成


17:24:44 | WARNING | 000333.SZ: 股东增减持采集失败 - (pymysql.err.ProgrammingError) nan can not be used with MySQL
[SQL: 
            INSERT INTO stock_holder_trade (`code`, `ann_date`, `holder_name`, `holder_type`, `trade_type`, `change_vol`, `change_ratio`, `after_share`, `after_ratio`, `avg_price`)
            VALUES (%(code)s, %(ann_date)s, %(holder_name)s, %(holder_type)s, %(trade_type)s, %(change_vol)s, %(change_ratio)s, %(after_share)s, %(after_ratio)s, %(avg_price)s)
            ON DUPLICATE KEY UPDATE `ann_date`=VALUES(`ann_date`), `holder_type`=VALUES(`holder_type`), `trade_type`=VALUES(`trade_type`), `change_vol`=VALUES(`change_vol`), `change_ratio`=VALUES(`change_ratio`), `after_share`=VALUES(`after_share`), `after_ratio`=VALUES(`after_ratio`), `avg_price`=VALUES(`avg_price`)
        ]
[parameters: [{'code': '000333.SZ', 'ann_date': '20260110', 'holder_name': '美的集团股份有限公司第八期全球合伙人', 'holder_type': 'C', 'trade_type': 'DE', 'change_vol': 3770433.0, 'change_ratio': 0.0551, 'after_share':

待采集: ['000333.SZ', '000858.SZ', '300750.SZ', '600519.SH', '601318.SH']


17:24:45 | WARNING | 000858.SZ: 股东增减持采集失败 - (pymysql.err.ProgrammingError) nan can not be used with MySQL
[SQL: 
            INSERT INTO stock_holder_trade (`code`, `ann_date`, `holder_name`, `holder_type`, `trade_type`, `change_vol`, `change_ratio`, `after_share`, `after_ratio`, `avg_price`)
            VALUES (%(code)s, %(ann_date)s, %(holder_name)s, %(holder_type)s, %(trade_type)s, %(change_vol)s, %(change_ratio)s, %(after_share)s, %(after_ratio)s, %(avg_price)s)
            ON DUPLICATE KEY UPDATE `ann_date`=VALUES(`ann_date`), `holder_type`=VALUES(`holder_type`), `trade_type`=VALUES(`trade_type`), `change_vol`=VALUES(`change_vol`), `change_ratio`=VALUES(`change_ratio`), `after_share`=VALUES(`after_share`), `after_ratio`=VALUES(`after_ratio`), `avg_price`=VALUES(`avg_price`)
        ]
[parameters: [{'code': '000858.SZ', 'ann_date': '20251011', 'holder_name': '四川省宜宾五粮液集团有限公司', 'holder_type': 'C', 'trade_type': 'IN', 'change_vol': 1509600.0, 'change_ratio': 0.0389, 'after_share': 801

17:24:45 | INFO    | 股东增减持采集完成: 5 只, 共 18 条


股东增减持采集完成: 18 条


## 2. 股东户数

In [ ]:
n = collector.collect_holder_count(stock_codes)
print(f"股东户数采集完成: {n} 条")

17:24:46 | INFO    | 股东户数采集完成: 5 只, 共 92 条


股东户数采集完成: 92 条


## 3. 数据验证

In [ ]:
from invest_model.repositories.event_repo import EventRepository

event_repo = EventRepository(engine)

# 查看贵州茅台股东增减持
ht = event_repo.get_holder_trade("600519.SH")
print(f"600519.SH 股东增减持 ({len(ht)} 条):")
if not ht.empty:
    print(ht.head(10).to_string(index=False))

print("\n")

# 查看股东户数
hc = event_repo.get_holder_count("600519.SH")
print(f"600519.SH 股东户数 ({len(hc)} 条):")
if not hc.empty:
    print(hc.tail(10).to_string(index=False))

600519.SH 股东增减持 (6 条):
 id      code ann_date        holder_name holder_type trade_type  change_vol  change_ratio  after_share  after_ratio avg_price begin_date close_date          created_at
 12 600519.SH 20251230 中国贵州茅台酒厂(集团)有限责任公司           C         IN   2003538.0        0.1600  681282935.0      54.4038      None       None       None 2026-04-07 17:24:45
 13 600519.SH 20250902 中国贵州茅台酒厂(集团)有限责任公司           C         IN     67821.0        0.0054  679279397.0      54.2438      None       None       None 2026-04-07 17:24:45
 14 600519.SH 20230628 中国贵州茅台酒厂(集团)有限责任公司           C         IN    771291.0        0.0614  679211576.0      54.0688      None       None       None 2026-04-07 17:24:45
 15 600519.SH 20230628 贵州茅台酒厂(集团)技术开发有限公司           C         IN     31400.0        0.0025   27849688.0       2.2170      None       None       None 2026-04-07 17:24:45
 16 600519.SH 20230211 中国贵州茅台酒厂(集团)有限责任公司           C         IN    148330.0        0.0118  678440285.0      54.0074      None      

## 完成

事件数据采集完毕，继续 `07_market_data.ipynb` 采集市场级别数据。